In [1]:
import torch
import esm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import json
import argparse
from datetime import datetime
from collections import defaultdict

In [2]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
model.eval()

ESM2(
  (embed_tokens): Embedding(33, 320, padding_idx=1)
  (layers): ModuleList(
    (0-5): 6 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=320, out_features=320, bias=True)
        (v_proj): Linear(in_features=320, out_features=320, bias=True)
        (q_proj): Linear(in_features=320, out_features=320, bias=True)
        (out_proj): Linear(in_features=320, out_features=320, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=320, out_features=1280, bias=True)
      (fc2): Linear(in_features=1280, out_features=320, bias=True)
      (final_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=120, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((320,), eps=1e-05, elementwis

In [3]:
from datasets import load_dataset
dataset = load_dataset("lightonai/SwissProt-EC-leaf", split="train")

In [10]:
count = 0
subset = []
for temp in dataset:
    if len(temp['seq']) <= 512 and count < 500:
        subset.append(temp)
        count += 1

In [11]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
model.eval()

batch_converter = alphabet.get_batch_converter()
data = [("test_protein", subset[0]['seq'])]
batch_labels, batch_strs, batch_tokens = batch_converter(data)


with torch.no_grad():
    results = model(batch_tokens, repr_layers=[5],return_contacts=False)

representations = results["representations"][5]
print("Full output shape:", representations.shape)

Full output shape: torch.Size([1, 468, 320])


In [12]:
results = model(batch_tokens, repr_layers=[1, 2, 3, 4, 5, 6], return_contacts=False)

for layer in [1, 2, 3, 4, 5, 6]:
    repr = results["representations"][layer]
    print(f"Layer {layer}: {repr.shape}")

Layer 1: torch.Size([1, 468, 320])
Layer 2: torch.Size([1, 468, 320])
Layer 3: torch.Size([1, 468, 320])
Layer 4: torch.Size([1, 468, 320])
Layer 5: torch.Size([1, 468, 320])
Layer 6: torch.Size([1, 468, 320])


In [13]:
class SparseAutoencoder(nn.Module):
    """Standard SAE with ReLU activation and L1 sparsity penalty."""

    def __init__(self, input_dim=320, hidden_dim=1280, l1_coeff=0.3):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.l1_coeff = l1_coeff
        self.sae_type = 'standard'

        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=True)

        nn.init.kaiming_uniform_(self.encoder.weight, nonlinearity='relu')
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.zeros_(self.decoder.bias)

    def encode(self, x):
        return F.relu(self.encoder(x))

    def decode(self, features):
        return self.decoder(features)

    def forward(self, x):
        features = self.encode(x)
        reconstruction = self.decode(features)
        return reconstruction, features

    def compute_loss(self, x, reconstruction, features):
        recon_loss = F.mse_loss(reconstruction, x)
        sparsity_loss = torch.mean(torch.abs(features))
        total_loss = recon_loss + self.l1_coeff * sparsity_loss
        return total_loss, recon_loss, sparsity_loss

    @torch.no_grad()
    def get_decoder_norms(self):
        return torch.norm(self.decoder.weight, dim=0)


In [14]:
input_dim = 320
hidden_dim = 1280

In [15]:
all_residue_vectors = []

for i, protein in enumerate(subset):
    data = [(f"protein_{i}", protein['seq'])]
    _, _, tokens = batch_converter(data)
    
    with torch.no_grad():
        results = model(tokens, repr_layers=[5], return_contacts=False)
    
    seq_len = len(protein['seq'])
    residue_reprs = results["representations"][5][0, 1:seq_len+1, :]
    all_residue_vectors.append(residue_reprs)

all_residue_vectors = torch.cat(all_residue_vectors, dim=0)
print("SAE training data shape:", all_residue_vectors.shape)

SAE training data shape: torch.Size([152304, 320])


In [17]:
train_loader = DataLoader(all_residue_vectors, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(all_residue_vectors, batch_size=8, num_workers=4)
test_loader = DataLoader(all_residue_vectors, batch_size=8, num_workers=4)

In [18]:
sae = SparseAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim, l1_coeff=0.3)

In [19]:
device='cpu'
num_epochs = 10

In [20]:
sae = sae.to(device)
optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [21]:
@torch.no_grad()
def evaluate_sae(sae, data_loader, device='cpu'):
    """Evaluate SAE on a dataset. Returns aggregated metrics."""
    sae.eval()
    metrics = defaultdict(float)
    num_batches = 0

    for batch in data_loader:
        x = batch
        x = x.to(device)

        reconstruction, features = sae(x)
        total_loss, recon_loss, sparsity_loss = sae.compute_loss(x, reconstruction, features)
        l0 = (features > 0).float().sum(dim=1).mean().item()

        metrics['total_loss'] += total_loss.item()
        metrics['recon_loss'] += recon_loss.item()
        metrics['sparsity_loss'] += sparsity_loss.item()
        metrics['l0_sparsity'] += l0
        num_batches += 1

    return {k: v / num_batches for k, v in metrics.items()}

In [ ]:
history = {
        'train_total_loss': [], 'train_recon_loss': [], 'train_sparsity_loss': [],
        'train_l0_sparsity': [], 'train_active_features': [],
        'val_total_loss': [], 'val_recon_loss': [], 'val_l0_sparsity': [],
    }

best_val_loss = float('inf')
patience_counter = 0
best_state = None
sparsity_label = "Aux"
early_stopping_patience=10

for epoch in range(num_epochs):
    sae.train()
    epoch_metrics = defaultdict(float)
    num_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}')
    for data in train_loader:
        data = data.to(device)
        reconstruction, features = sae(data)
        # print(reconstruction.shape)
        # print(features.shape)
        # print('--')
        
    
        total_loss, recon_loss, sparsity_loss = sae.compute_loss(data, reconstruction, features)

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(sae.parameters(), max_norm=1.0)
        optimizer.step()

        with torch.no_grad():
            l0 = (features > 0).float().sum(dim=1).mean().item()
            active_features_total = (features > 0).any(dim=0).sum().item()

        epoch_metrics['total'] += total_loss.item()
        epoch_metrics['recon'] += recon_loss.item()
        epoch_metrics['sparsity'] += sparsity_loss.item()
        epoch_metrics['l0'] += l0
        epoch_metrics['active'] += active_features_total
        num_batches += 1

        pbar.set_postfix({
            'loss': f'{total_loss.item():.4f}',
            'recon': f'{recon_loss.item():.4f}',
            'L0': f'{l0:.1f}',
        })

    history['train_total_loss'].append(epoch_metrics['total'] / num_batches)
    history['train_recon_loss'].append(epoch_metrics['recon'] / num_batches)
    history['train_sparsity_loss'].append(epoch_metrics['sparsity'] / num_batches)
    history['train_l0_sparsity'].append(epoch_metrics['l0'] / num_batches)
    history['train_active_features'].append(epoch_metrics['active'] / num_batches)

    if val_loader is not None:
        val_metrics = evaluate_sae(sae, val_loader, device)
        history['val_total_loss'].append(val_metrics['total_loss'])
        history['val_recon_loss'].append(val_metrics['recon_loss'])
        history['val_l0_sparsity'].append(val_metrics['l0_sparsity'])

        scheduler.step(val_metrics['total_loss'])

        if val_metrics['total_loss'] < best_val_loss:
            best_val_loss = val_metrics['total_loss']
            best_state = {k: v.clone() for k, v in sae.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    print(f"Epoch {epoch + 1} — "
          f"Train Loss: {history['train_total_loss'][-1]:.4f}, "
          f"Recon: {history['train_recon_loss'][-1]:.6f}, "
          f"L0: {history['train_l0_sparsity'][-1]:.1f}/{sae.hidden_dim}, "
          f"{sparsity_label}: {history['train_sparsity_loss'][-1]:.6f}"
          + (f", Val Loss: {history['val_total_loss'][-1]:.4f}" if val_loader else ""))

Epoch 1/10:   0%|                                                                      | 0/19038 [01:23<?, ?it/s, loss=0.1832, recon=0.1111, L0=319.2]

Epoch 1 — Train Loss: 0.3004, Recon: 0.203408, L0: 338.3/1280, Aux: 0.323355, Val Loss: 0.1942



Epoch 1/10:   0%|                                                                      | 0/19038 [01:34<?, ?it/s, loss=0.1832, recon=0.1111, L0=319.2]

Epoch 2/10:   0%|                                                                      | 0/19038 [00:04<?, ?it/s, loss=0.1586, recon=0.0939, L0=313.1]